## AdaBoost - full derivation

Sequentially fit weak learners $h_t(x)\in\{-1,+1\}$, reweighting
misclassified points up each round. Final model:
$H(x)=\text{sign}\left(\sum_t \alpha_t h_t(x)\right)$.

**Derivation of $\alpha_t$:** AdaBoost minimizes the exponential loss
$L=\sum_i e^{-y_i H(x_i)}$. At round $t$, given the accumulated model
$H_{t-1}$, we add $\alpha_t h_t$ to minimize:
$$\sum_i w_i^{(t)} e^{-\alpha_t y_i h_t(x_i)}, \qquad w_i^{(t)}=e^{-y_i H_{t-1}(x_i)}$$
Split the sum into correctly-classified ($y_ih_t(x_i)=1$) and
misclassified ($=-1$) points, let $\epsilon_t=\frac{\sum_{i:\text{wrong}} w_i^{(t)}}{\sum_i w_i^{(t)}}$
be the weighted error rate. Differentiating w.r.t. $\alpha_t$ and setting
to zero gives:
$$\boxed{\alpha_t = \frac12\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)}$$
and the weight update (dropping the normalizing constant):
$$w_i^{(t+1)} = w_i^{(t)}\cdot e^{-\alpha_t y_i h_t(x_i)}$$
- i.e. multiply misclassified points' weight by $e^{\alpha_t}>1$ and
correct ones by $e^{-\alpha_t}<1$.

### Worked numerical example

5 points, all weight $w_i=0.2$ initially. Weak learner $h_1$ misclassifies
1 of them. $\epsilon_1 = 0.2/1.0 = 0.2$.
$$\alpha_1 = \frac12\ln\left(\frac{0.8}{0.2}\right)=\frac12\ln4=\frac12(1.386)=0.693$$
Update weights: misclassified point → $0.2\times e^{0.693}=0.2\times2.0=0.4$.
Correctly classified points → $0.2\times e^{-0.693}=0.2\times0.5=0.1$ each.
New unnormalized weights: $[0.1,0.1,0.1,0.1,0.4]$, sum$=0.8$; normalize by
dividing by 0.8: $[0.125,0.125,0.125,0.125,0.5]$ - the misclassified point
now gets **4×** the attention of the others in round 2.

## Python - verify the AdaBoost hand-worked example

In [1]:
import numpy as np

w = np.array([0.2]*5)
misclassified = np.array([False, False, False, False, True])
eps1 = w[misclassified].sum() / w.sum()
alpha1 = 0.5 * np.log((1-eps1)/eps1)
print(eps1, alpha1)   # 0.2  0.6931...

new_w = np.where(misclassified, w*np.exp(alpha1), w*np.exp(-alpha1))
new_w /= new_w.sum()
print(new_w)   # [0.125 0.125 0.125 0.125 0.5]

# Gradient boosting residual check
y = np.array([2.0, 2.5, 4.5, 5.0])
F0 = y.mean()
resid1 = y - F0
print(F0, resid1)   # 3.5  [-1.5 -1.  1.  1.5]

0.2 0.6931471805599453
[0.125 0.125 0.125 0.125 0.5  ]
3.5 [-1.5 -1.   1.   1.5]
